In [ ]:
%pip install -q zarr dask distributed cftime netCDF4 gcsfs

from pathlib import Path

# Set to True to use Google Drive; local Colab storage is the default.
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/riskclima")
else:
    BASE_DIR = Path("/content/riskclima")

# Edit BASE_DIR above if your data or outputs use another neutral location.
BASE_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)


In [ ]:
from tqdm.auto import tqdm
import math

# ERA5 ARCO/Zarr
DATASET_ID = "ERA5_ARCO"
SOURCE = "era5"
VARIABLE_T2M = "t2m"
VARIABLE_D2M = "d2m"

# ERA5 calibration
CALIBRATION_PERIOD = ("1961-01-01", "1990-12-31")

# Slice
LAT_SLICE = slice(-70, 20)
LON_SLICE = slice(-120, -5)

# Output
OUTPUT_DIR = BASE_DIR / "ERA5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MONTHLY_DIR = OUTPUT_DIR / "monthly"
MONTHLY_DIR.mkdir(parents=True, exist_ok=True)

MONTHLY_PARTS_DIR = MONTHLY_DIR / "parts"
MONTHLY_PARTS_DIR.mkdir(parents=True, exist_ok=True)

MONTHLY_OUTPUT_FILE = MONTHLY_DIR / "xhwi_era5_monthly_ind_prod.nc"
CALIB_OUT = MONTHLY_DIR / "xhwi_era5_calib_t2m_max_1961-1990.nc"

DEFAULT_MONTHS = list(range(1, 13))

# Global variables
TORCH_DTYPE = "float32"
XHWI_MINIMUM = 0.001

# Tune these values in Colab depending on available GPU RAM.
LAT_BLOCK_SIZE = 64
LON_BLOCK_SIZE = 64

# Months to Run
MONTHS_TO_RUN = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Calibration:", CALIBRATION_PERIOD)


In [ ]:
from datetime import datetime, timezone

import gc
import numpy as np
import xarray as xr
import torch
import gcsfs
from dask.diagnostics import ProgressBar

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float32 if TORCH_DTYPE == 'float32' else torch.float64

print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(torch.cuda.get_device_name(0))


In [ ]:
# era5/spatial/scripts/src/preprocessing/era5.py

import os

def standardize_era5_dims(ds: xr.Dataset) -> xr.Dataset:
    """Rename ERA5 coordinates to lat/lon if needed and sort dimensions."""
    rename = {}

    if "latitude" in ds.dims or "latitude" in ds.coords:
        rename["latitude"] = "lat"
    if "longitude" in ds.dims or "longitude" in ds.coords:
        rename["longitude"] = "lon"
    if "valid_time" in ds.dims or "valid_time" in ds.coords:
        rename["valid_time"] = "time"

    if rename:
        ds = ds.rename(rename)

    for dim in ["time", "lat", "lon"]:
        if dim in ds.dims:
            ds = ds.sortby(dim)

    return ds


def get_cdsapi_key() -> str:
    # 1) Env Var
    key = os.getenv("CDSAPI_KEY")
    if key and key.strip():
        return key.strip()

    # 2) Colab Secrets
    try:
        from google.colab import userdata
        key = userdata.get("CDSAPI_KEY")
        if key and key.strip():
            return key.strip()
    except Exception:
        pass

    raise ValueError(
        "CDS API key not found. Set CDSAPI_KEY or add it to Colab Secrets."
    )


def open_era5_zarr(path: str, chunks="auto") -> xr.Dataset:
    cdsapi_key = get_cdsapi_key()

    ds = xr.open_zarr(
        path,
        consolidated=True,
        chunks=chunks,
        storage_options={
            "headers": {"Authorization": f"Bearer {cdsapi_key}"}
        },
    )

    return standardize_era5_dims(ds)


def open_saved_tasmax_calibration(calibration_path: Path | str) -> xr.DataArray:
    da = xr.open_dataarray(calibration_path)

    da = standardize_era5_dims(da)

    if "time" in da.dims:
        da = da.rename({"time": "calibration_time"})

    if "calibration_time" not in da.dims:
        raise ValueError("Saved calibration file must have 'calibration_time' dimension.")

    if "lat" not in da.dims or "lon" not in da.dims:
        raise ValueError("Saved calibration file must have 'lat' and 'lon' dimensions.")

    return da


def kelvin_to_celsius(da: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get("units", "")).lower()
    out = da - 273.15 if units in {"k", "kelvin"} else da
    out.attrs = da.attrs.copy()
    out.attrs["units"] = "degC"
    return out


In [ ]:
# era5/spatial/scripts/src/features/humidity.py

def dewpoint_to_relative_humidity(
    d2m: xr.DataArray,
    t2m: xr.DataArray,
    clip: bool = True,
) -> xr.DataArray:
    """Calculate relative humidity (%) from 2 m dewpoint and 2 m temperature.

    Inputs must be in Kelvin.
    Formula follows ECMWF IFS transformation:
    RH = 100 * F(Td) / F(T)
    """
    rh = 100.0 * np.exp(
        (17.502 * (d2m - 273.16) / (d2m - 32.19)) -
        (17.502 * (t2m - 273.16) / (t2m - 32.19))
    )

    if clip:
        rh = rh.clip(min=0.0, max=100.0)

    rh.name = "hurs"
    rh.attrs.update({
        "long_name": "Relative humidity calculated from 2 m dewpoint temperature and 2 m temperature",
        "units": "%",
        "formula": "RH = 100 * exp(17.502*(d2m-273.16)/(d2m-32.19) - 17.502*(t2m-273.16)/(t2m-32.19))",
        "source_variables": "d2m, t2m",
    })

    return rh


In [ ]:
# era5/spatial/scripts/src/torch_ops/cdf.py

def torch_match_cdf_linear(
    tas_hourly_c: torch.Tensor,
    tasmax_calibration_c: torch.Tensor,
) -> torch.Tensor:
    """Match hourly tas to empirical calibration CDF using linear interpolation."""
    time_size, y_size, x_size = tas_hourly_c.shape

    values = tas_hourly_c.reshape(time_size, -1).transpose(0, 1).contiguous()
    calibration = tasmax_calibration_c.reshape(tasmax_calibration_c.shape[0], -1).transpose(0, 1).contiguous()

    finite_cal = torch.isfinite(calibration)
    n_valid = finite_cal.sum(dim=1)
    calibration_sorted = calibration.masked_fill(~finite_cal, float('inf')).sort(dim=1).values

    finite_values = torch.isfinite(values)
    safe_values = values.masked_fill(~finite_values, 0.0)

    idx_right = torch.searchsorted(calibration_sorted, safe_values, right=False)
    max_idx = torch.clamp(n_valid - 1, min=0).unsqueeze(1)
    idx1 = torch.minimum(idx_right, max_idx).long()
    idx0 = torch.clamp(idx1 - 1, min=0).long()

    x0 = calibration_sorted.gather(1, idx0)
    x1 = calibration_sorted.gather(1, idx1)

    n_valid_f = n_valid.clamp(min=1).unsqueeze(1).to(values.dtype)
    y0 = (idx0.to(values.dtype) + 1.0) / n_valid_f
    y1 = (idx1.to(values.dtype) + 1.0) / n_valid_f

    denom = x1 - x0
    frac = torch.where(torch.abs(denom) > 0, (safe_values - x0) / denom, torch.zeros_like(safe_values))
    target = y0 + frac * (y1 - y0)

    first = calibration_sorted[:, 0].unsqueeze(1)
    last = calibration_sorted.gather(1, max_idx.long())
    target = torch.where(safe_values < first, torch.zeros_like(target), target)
    target = torch.where(safe_values >= last, torch.ones_like(target), target)
    target = torch.where((n_valid < 2).unsqueeze(1), torch.full_like(target, float('nan')), target)
    target = torch.where(finite_values, target, torch.full_like(target, float('nan')))
    target = torch.clamp(target, min=0.0, max=1.0)

    return target.transpose(0, 1).reshape(time_size, y_size, x_size)


In [ ]:
# era5/spatial/scripts/src/torch_ops/xhwi.py

def torch_heatwave_index(
    tas_c: torch.Tensor,
    hurs: torch.Tensor,
    target: torch.Tensor,
) -> torch.Tensor:
    target100 = target * 100.0
    tpe = torch.clamp(target100 - 95.0, min=0.0)
    coef = (torch.exp(tpe) * hurs) / 1000.0
    xhwi = (coef - 0.001) / 14.84

    xhwi = torch.where(tpe > 0, xhwi, torch.zeros_like(xhwi))
    xhwi = torch.where(tas_c > 32.0, xhwi, torch.zeros_like(xhwi))
    xhwi = torch.where(xhwi > XHWI_MINIMUM, xhwi, torch.zeros_like(xhwi))
    return xhwi


In [ ]:
# era5/spatial/scripts/src/torch_ops/aggregations.py

def month_keys_from_time(time_coord: xr.DataArray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    years = time_coord.dt.year.values.astype(np.int64)
    months = time_coord.dt.month.values.astype(np.int64)
    days = time_coord.dt.day.values.astype(np.int64)
    day_keys = years * 10000 + months * 100 + days
    month_keys = years * 100 + months
    return day_keys, month_keys, np.asarray(time_coord.values)


def torch_monthly_accumulated_xhwi(
    xhwi: torch.Tensor,
    time_coord: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray]:
    """Aggregate hourly XHWI into monthly accumulated values on GPU."""
    day_keys, month_keys_by_time, time_values = month_keys_from_time(time_coord)
    unique_days = np.unique(day_keys)

    daily_values = []
    daily_month_keys = []

    for day_key in unique_days:
        idx_np = np.flatnonzero(day_keys == day_key)
        idx = torch.as_tensor(idx_np, device=xhwi.device, dtype=torch.long)
        xhwi_day = xhwi.index_select(0, idx)
        active_hours = (xhwi_day != 0).sum(dim=0).to(xhwi.dtype)
        daily_sum = xhwi_day.sum(dim=0)
        daily_values.append(active_hours * daily_sum)
        daily_month_keys.append(month_keys_by_time[idx_np[0]])

    if not daily_values:
        raise ValueError('No daily values were generated for this block.')

    daily_stack = torch.stack(daily_values, dim=0)
    daily_month_keys = np.asarray(daily_month_keys)
    unique_months = np.unique(daily_month_keys)

    monthly_values = []
    monthly_time_values = []
    for month_key in unique_months:
        day_idx_np = np.flatnonzero(daily_month_keys == month_key)
        day_idx = torch.as_tensor(day_idx_np, device=xhwi.device, dtype=torch.long)
        monthly_values.append(daily_stack.index_select(0, day_idx).sum(dim=0))
        first_time_idx = np.flatnonzero(month_keys_by_time == month_key)[0]
        monthly_time_values.append(time_values[first_time_idx])

    monthly = torch.stack(monthly_values, dim=0).detach().cpu().numpy().astype('float32')
    return monthly, np.asarray(monthly_time_values)


In [ ]:
# era5/spatial/scripts/src/io/writers.py

def build_monthly_output_dataset(monthly: xr.DataArray) -> xr.Dataset:
    ds = monthly.to_dataset(name='xhwi_monthly_accumulated')
    creation_date = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

    time_start = np.datetime_as_string(ds['time'].values[0], unit='D') if 'time' in ds.coords and ds.sizes.get('time', 0) else 'unknown'
    time_end = np.datetime_as_string(ds['time'].values[-1], unit='D') if 'time' in ds.coords and ds.sizes.get('time', 0) else 'unknown'
    lat_min = float(ds['lat'].min()) if 'lat' in ds.coords else float('nan')
    lat_max = float(ds['lat'].max()) if 'lat' in ds.coords else float('nan')
    lon_min = float(ds['lon'].min()) if 'lon' in ds.coords else float('nan')
    lon_max = float(ds['lon'].max()) if 'lon' in ds.coords else float('nan')

    if 'time' in ds.coords:
        ds['time'].attrs.update({
            'standard_name': 'time',
            'long_name': 'Time',
            'axis': 'T',
        })

    if 'lat' in ds.coords:
        ds['lat'].attrs.update({
            'standard_name': 'latitude',
            'long_name': 'Latitude',
            'units': 'degrees_north',
            'axis': 'Y',
        })

    if 'lon' in ds.coords:
        ds['lon'].attrs.update({
            'standard_name': 'longitude',
            'long_name': 'Longitude',
            'units': 'degrees_east',
            'axis': 'X',
        })

    ds['xhwi_monthly_accumulated'].attrs.update({
        'long_name': 'Monthly accumulated Extreme Heatwave Index',
        'units': '1',
        'cell_methods': 'time: sum',
        'description': (
            'Monthly sum of daily XHWI products. Each daily product is the number '
            'of hours with nonzero XHWI multiplied by the daily sum of hourly XHWI.'
        ),
    })

    ds.attrs.update({
        'title': 'ERA5 - monthly accumulated Extreme Heatwave Index (XHWI)',
        'summary': (
            'Subdomain calculation of the monthly accumulated Extreme Heatwave Index '
            f'from ERA5 single-level data. Spatial domain: lat [{lat_min:.2f} deg, {lat_max:.2f} deg], '
            f'lon [{lon_min:.2f} deg, {lon_max:.2f} deg]. Temporal coverage: {time_start} to {time_end}. '
            'The index is computed from hourly 2 m temperature and relative humidity derived from 2 m dewpoint temperature. '
            'The calibration CDF is calendar-month-specific and uses daily maximum 2 m temperature derived from hourly ERA5 t2m for 1961-1990. '
            'Prepared for use in the RiskClima climate-risk index pipelines.'
        ),
        'keywords': 'xhwi, extreme heatwave index, ERA5, reanalysis, heatwave, climate risk, South America, RiskClima',
        'source': (
            'ERA5 hourly data on single levels from the Copernicus Climate Data Store analysis-ready cloud-optimised Zarr store. '
            'Variables used: 2 m temperature (t2m) and 2 m dewpoint temperature (d2m). '
            'Original zarr: reanalysis_era5_single_levels/sfc/geoChunked.zarr.'
        ),
        'history': f'{creation_date} Computed monthly accumulated XHWI from ERA5 ARCO Zarr using xarray and a PyTorch blockwise implementation.',
        'creation_date': creation_date,
        'creator': 'RiskClima project contributors',
        'references': 'https://riskclima.com.br/',
        'code_repository': 'https://github.com/lammoc-uff/cnpq-riskclima',
        'institution': 'Climate System Monitoring and Modeling Laboratory (LAMMOC), Universidade Federal Fluminense (UFF), Niteroi, Brazil',
        'project': 'RiskClima',
        'license': 'CC-BY-4.0',
        'Conventions': 'CF-1.10',
        'processing_level': 'Processed data',
        'dataset_id': DATASET_ID,
        'source_id': SOURCE,
        'calibration_period': f'{CALIBRATION_PERIOD[0]} to {CALIBRATION_PERIOD[1]}',
        'calibration_method': 'Separate empirical CDF for each calendar month and grid cell.',
        'compute_backend': f'PyTorch on {DEVICE.type}',
        'input_variables': f'{VARIABLE_T2M}, {VARIABLE_D2M}',
        'humidity_method': 'Relative humidity calculated from 2 m dewpoint temperature and 2 m temperature.',
        'temperature_threshold_c': 32.0,
        'cdf_threshold_percent': 95.0,
        'xhwi_minimum': XHWI_MINIMUM,
        'scientific_profile': 'xhwi-2024-v1',
        'comment': 'Monthly accumulated XHWI computed from ERA5 ARCO data. Calibration is based on 1961-1990 daily maximum t2m, separately for each calendar month and grid cell.',
    })

    return ds


def normalize_months(months: list[int] | tuple[int, ...] | None = None) -> list[int]:
    if months is None:
        months = DEFAULT_MONTHS

    months = [int(month) for month in months]

    invalid_months = [month for month in months if month < 1 or month > 12]
    if invalid_months:
        raise ValueError(f"Invalid months: {invalid_months}. Expected values from 1 to 12.")

    if len(months) != len(set(months)):
        raise ValueError(f"Duplicated months found: {months}")

    return sorted(months)


def months_tag(months: list[int] | tuple[int, ...] | None = None) -> str:
    months = normalize_months(months)

    if months == DEFAULT_MONTHS:
        return "01-12"

    return "-".join(f"{month:02d}" for month in months)


def monthly_part_path(months: list[int] | tuple[int, ...] | None = None) -> Path:
    months = normalize_months(months)

    if len(months) == 1:
        filename = f"xhwi_era5_month_{months[0]:02d}.nc"
    else:
        filename = f"xhwi_era5_months_{months_tag(months)}.nc"

    return MONTHLY_PARTS_DIR / filename


def write_monthly_netcdf(
    ds: xr.Dataset,
    output_path: Path | str | None = None,
    overwrite: bool = True,
) -> Path:
    output_path = Path(output_path) if output_path is not None else MONTHLY_OUTPUT_FILE
    output_path.parent.mkdir(parents=True, exist_ok=True)

    ds = ds.sortby("time")

    if "time" in ds.indexes and not ds.indexes["time"].is_monotonic_increasing:
        raise ValueError("Output time coordinate is not monotonic increasing.")

    if "time" in ds.indexes and not ds.indexes["time"].is_unique:
        raise ValueError("Output time coordinate contains duplicated values.")

    encoding = {
        "xhwi_monthly_accumulated": {
            "zlib": True,
            "complevel": 4,
            "_FillValue": np.float32(np.nan),
            "dtype": "float32",
        }
    }

    if output_path.exists():
        if overwrite:
            output_path.unlink()
        else:
            raise FileExistsError(f"File already exists: {output_path}")

    with ProgressBar():
        ds.to_netcdf(output_path, engine="netcdf4", encoding=encoding)

    return output_path


def concat_monthly_netcdfs(
    input_paths: list[Path | str] | None = None,
    output_path: Path | str | None = None,
    overwrite: bool = True,
) -> Path:
    if input_paths is None:
        input_paths = sorted(MONTHLY_PARTS_DIR.glob("xhwi_era5_month*.nc"))

    input_paths = [Path(path) for path in input_paths]

    if not input_paths:
        raise ValueError(f"No monthly part files found in {MONTHLY_PARTS_DIR}")

    print("Files to concatenate:")
    for path in input_paths:
        print(path)

    ds_raw = xr.open_mfdataset(
        input_paths,
        combine="nested",
        concat_dim="time",
        engine="netcdf4",
    )

    ds_raw = ds_raw.sortby("time")

    if "time" in ds_raw.indexes and not ds_raw.indexes["time"].is_unique:
        duplicated_times = ds_raw.indexes["time"][ds_raw.indexes["time"].duplicated()]
        raise ValueError(
            f"Duplicated time values found during concatenation. "
            f"First duplicated values: {duplicated_times[:10]}"
        )

    monthly = ds_raw["xhwi_monthly_accumulated"]

    ds_final = build_monthly_output_dataset(monthly)
    ds_final.attrs["source_monthly_part_files"] = "; ".join(str(path) for path in input_paths)

    output_path = Path(output_path) if output_path is not None else MONTHLY_OUTPUT_FILE

    written_path = write_monthly_netcdf(
        ds_final,
        output_path=output_path,
        overwrite=overwrite,
    )

    ds_raw.close()

    return written_path

In [ ]:
# era5/spatial/scripts/src/pipeline/data_access.py

def get_era5_variable(ds: xr.Dataset, long_name: str, short_name: str) -> xr.DataArray:
    """Get ERA5 variable accepting either ARCO long names or ERA5 short names."""
    if long_name in ds:
        return ds[long_name]
    if short_name in ds:
        return ds[short_name]
    raise KeyError(f"Variable not found. Tried: {long_name}, {short_name}")


def open_era5_inputs(path: str) -> tuple[xr.DataArray, xr.DataArray]:

    ds = open_era5_zarr(path, chunks="auto")

    ds = ds.sel(
        lat=LAT_SLICE,
        lon=LON_SLICE,
    )

    t2m = get_era5_variable(ds, VARIABLE_T2M, "t2m")
    d2m = get_era5_variable(ds, VARIABLE_D2M, "d2m")

    t2m_c = kelvin_to_celsius(t2m)
    hurs = dewpoint_to_relative_humidity(d2m=d2m, t2m=t2m)

    return t2m_c, hurs


def open_t2mcalib_inputs(path: str) -> xr.DataArray:

    ds = open_era5_zarr(path, chunks="auto")

    ds = ds.sel(
        lat=LAT_SLICE,
        lon=LON_SLICE,
    )

    t2m = get_era5_variable(ds, VARIABLE_T2M, "t2m")

    t2m_c = kelvin_to_celsius(t2m)

    return t2m_c


def open_calibration_tasmax_from_t2m(t2m_c: xr.DataArray) -> xr.DataArray:
    """Compute daily maximum 2 m temperature from hourly ERA5 t2m for calibration."""
    t2m_cal = t2m_c.sel(time=slice(*CALIBRATION_PERIOD))

    if t2m_cal.sizes.get("time", 0) == 0:
        raise ValueError(f"No t2m data found for calibration period {CALIBRATION_PERIOD}.")

    # ERA5 hourly regular data: 24 timesteps = 1 day
    tasmax = t2m_cal.coarsen(time=24, boundary="trim").max()

    # daily timestamp = first hour of each 24h block
    daily_time = t2m_cal["time"].isel(time=slice(0, None, 24))
    daily_time = daily_time.isel(time=slice(0, tasmax.sizes["time"]))

    tasmax = tasmax.assign_coords(time=daily_time)

    return tasmax.rename({"time": "calibration_time"})


def iter_spatial_blocks(lat_size: int, lon_size: int, lat_block: int, lon_block: int):
    for lat_start in range(0, lat_size, lat_block):
        lat_stop = min(lat_start + lat_block, lat_size)
        for lon_start in range(0, lon_size, lon_block):
            lon_stop = min(lon_start + lon_block, lon_size)
            yield slice(lat_start, lat_stop), slice(lon_start, lon_stop)

In [ ]:
# era5/spatial/scripts/src/pipeline/block_processor.py

def load_block_np(da: xr.DataArray, lat_slice: slice, lon_slice: slice) -> np.ndarray:
    return da.isel(lat=lat_slice, lon=lon_slice).load().values.astype('float32')


def process_month_block_torch(
    tas_c_month: xr.DataArray,
    hurs_month: xr.DataArray,
    tasmax_calibration_month: xr.DataArray,
    lat_slice: slice,
    lon_slice: slice,
) -> tuple[np.ndarray, np.ndarray]:
    tas_np = load_block_np(tas_c_month, lat_slice, lon_slice)
    hurs_np = load_block_np(hurs_month, lat_slice, lon_slice)
    tasmax_np = load_block_np(tasmax_calibration_month, lat_slice, lon_slice)

    tas_t = torch.as_tensor(tas_np, dtype=DTYPE, device=DEVICE)
    hurs_t = torch.as_tensor(hurs_np, dtype=DTYPE, device=DEVICE)
    tasmax_t = torch.as_tensor(tasmax_np, dtype=DTYPE, device=DEVICE)

    target_t = torch_match_cdf_linear(tas_t, tasmax_t)
    xhwi_t = torch_heatwave_index(tas_c=tas_t, hurs=hurs_t, target=target_t)
    monthly_np, monthly_time = torch_monthly_accumulated_xhwi(xhwi_t, tas_c_month['time'])

    del tas_t, hurs_t, tasmax_t, target_t, xhwi_t
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

    return monthly_np, monthly_time


In [ ]:
def compute_era5_monthly_xhwi_torch(
    path: str,
    calibration_path: Path | str | None = None,
    months: list[int] | tuple[int, ...] | None = None,
) -> xr.Dataset:
    months = normalize_months(months)

    print("Opening and preprocessing ERA5 inputs...")
    tas_c, hurs = open_era5_inputs(path)

    if calibration_path is None:
        print("Computing calibration tasmax from ERA5 t2m...")
        tasmax_calibration = open_calibration_tasmax_from_t2m(tas_c)
    else:
        print(f"Opening saved calibration file: {calibration_path}")
        tasmax_calibration = open_saved_tasmax_calibration(calibration_path)

    lat_values = tas_c["lat"].values
    lon_values = tas_c["lon"].values
    lat_size = tas_c.sizes["lat"]
    lon_size = tas_c.sizes["lon"]

    n_lat_blocks = math.ceil(lat_size / LAT_BLOCK_SIZE)
    n_lon_blocks = math.ceil(lon_size / LON_BLOCK_SIZE)
    n_blocks = n_lat_blocks * n_lon_blocks

    month_arrays = []
    month_times = []

    for month in tqdm(months, desc="Processing selected months", unit="month"):
        tqdm.write(f"Processing ERA5, calendar month {month:02d}...")

        tas_c_month = tas_c.sel(time=tas_c["time.month"] == month)
        hurs_month = hurs.sel(time=hurs["time.month"] == month)
        tasmax_calibration_month = tasmax_calibration.sel(
            calibration_time=tasmax_calibration["calibration_time.month"] == month
        )

        if tas_c_month.sizes.get("time", 0) == 0:
            tqdm.write(f"No hourly t2m data found for month {month:02d}; skipping.")
            continue

        if tasmax_calibration_month.sizes.get("calibration_time", 0) == 0:
            raise ValueError(f"No calibration tasmax data found for month {month}.")

        month_template = None
        month_time = None

        block_iter = iter_spatial_blocks(
            lat_size,
            lon_size,
            LAT_BLOCK_SIZE,
            LON_BLOCK_SIZE,
        )

        for lat_slice, lon_slice in tqdm(
            block_iter,
            total=n_blocks,
            desc=f"Month {month:02d} spatial blocks",
            unit="block",
            leave=False,
        ):
            block_np, block_time = process_month_block_torch(
                tas_c_month=tas_c_month,
                hurs_month=hurs_month,
                tasmax_calibration_month=tasmax_calibration_month,
                lat_slice=lat_slice,
                lon_slice=lon_slice,
            )

            if month_template is None:
                month_time = block_time
                month_template = np.full(
                    (len(month_time), lat_size, lon_size),
                    np.nan,
                    dtype="float32",
                )
            elif len(block_time) != len(month_time):
                raise ValueError("Inconsistent monthly time length across spatial blocks.")

            month_template[:, lat_slice, lon_slice] = block_np

        if month_template is not None:
            month_arrays.append(month_template)
            month_times.append(month_time)

    if not month_arrays:
        raise ValueError("No monthly XHWI outputs were generated for ERA5.")

    monthly_values = np.concatenate(month_arrays, axis=0)
    monthly_time = np.concatenate(month_times, axis=0)

    monthly = xr.DataArray(
        monthly_values,
        dims=("time", "lat", "lon"),
        coords={
            "time": monthly_time,
            "lat": lat_values,
            "lon": lon_values,
        },
        name="xhwi_monthly_accumulated",
    ).sortby("time")

    ds = build_monthly_output_dataset(monthly)
    ds.attrs["processed_calendar_months"] = ", ".join(f"{month:02d}" for month in months)

    return ds

In [ ]:
# Create the calibration needed by the monthly run.
ERA5_ZARR_PATH = (
    "https://arco.datastores.ecmwf.int/"
    "cadl-arco-geo-002/arco/reanalysis_era5_single_levels/sfc/geoChunked.zarr"
)

if CALIB_OUT.exists():
    print(f"Calibration file already exists: {CALIB_OUT}")
else:
    print("Opening ERA5 t2m calibration input...")
    tas_c_calibration = open_t2mcalib_inputs(ERA5_ZARR_PATH)
    print("Computing daily maximum t2m calibration field...")
    tasmax_calibration = open_calibration_tasmax_from_t2m(tas_c_calibration)
    tasmax_calibration.name = "tasmax_calibration"
    tasmax_calibration.attrs.update({
        "long_name": "Daily maximum 2 m temperature for XHWI calibration",
        "units": "degC",
        "source_variable": "t2m",
        "calculation": "daily maximum from hourly ERA5 t2m",
    })
    encoding = {
        "tasmax_calibration": {
            "zlib": True,
            "complevel": 4,
            "_FillValue": np.float32(np.nan),
            "dtype": "float32",
        }
    }
    print(f"Writing calibration file to: {CALIB_OUT}")
    with ProgressBar():
        tasmax_calibration.to_netcdf(
            CALIB_OUT,
            engine="netcdf4",
            encoding=encoding,
        )
    print(f"Calibration file written: {CALIB_OUT}")

In [ ]:
# Run selected months

ERA5_ZARR_PATH = (
    "https://arco.datastores.ecmwf.int/"
    "cadl-arco-geo-002/arco/reanalysis_era5_single_levels/sfc/geoChunked.zarr"
)

part_output = monthly_part_path(MONTHS_TO_RUN)

if part_output.exists():
    print(f"File already exists, skipping processing: {part_output}")
else:
    ds_era5_monthly = compute_era5_monthly_xhwi_torch(
        ERA5_ZARR_PATH,
        calibration_path=CALIB_OUT,
        months=MONTHS_TO_RUN,
    )

    display(ds_era5_monthly)

    written_path = write_monthly_netcdf(
        ds_era5_monthly,
        output_path=part_output,
    )

    print("Written:")
    print(written_path)

In [ ]:
# Concatenate monthly parts into final file

part_files = sorted(MONTHLY_PARTS_DIR.glob("xhwi_era5_month*.nc"))

final_output = concat_monthly_netcdfs(
    input_paths=part_files,
    output_path=MONTHLY_OUTPUT_FILE,
    overwrite=True,
)

print("Final output written:")
print(final_output)

ds_final_check = xr.open_dataset(final_output)
display(ds_final_check)